# 用 MCMC 样本直接重建 M、S、v（全部 K×K）

旧 notebook 为了讲原理，习惯性地枚举了全部 16 个构型、用 **完整 H（16×16）** 显式构造
$M=\sum_x \psi^\dagger H\psi$、$S=\sum_x \psi^\dagger\psi$。这造成一个错觉：好像"转回本征态"要先对角线化 16×16 的 H，成本连精确对角化都不如。

本 notebook 彻底改成**基于 MCMC 样本**的重建，全程只出现 **K×K** 矩阵：

- `M` : K×K（哈密顿量在 K 列子空间上的投影）
- `S` : K×K（K 列的重叠）
- `v` : K×K（广义本征问题的旋转系数）

关键恒等式（每个 walker 都成立）：
$$
\big(P^\dagger H P,\; P^\dagger P\big)_{\text{K×K 广义本征}}
\Longleftrightarrow
\underbrace{P^{-1} H P}_{\text{训练的 }E_L\text{ 矩阵，K×K}},\quad \text{本征值相等}
$$

$$
(P^\dagger P)^{-1}(P^\dagger H P) = P^{-1}(P^{\dagger})^{-1} P^\dagger H P = P^{-1} H P
$$

其中单 walker 矩阵 $P_{\alpha\beta}=\psi_\beta(x_\alpha)$（K×K，把 $\beta$ 列拟设算到 $\alpha$ 个副本构型上），$H\psi$ 用训练的局部能量算子 $\texttt{Ham\_Psi\_scaled}$ 作用在**被采样的构型**上（不触碰完整 H 矩阵）。


## 0. 载入环境与训练历史

In [1]:
import pickle
import numpy as np
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
from scipy.linalg import eigh

from NES_VMC_V1 import (
    NESTotalAnsatz_stable,
    create_single_machine_gauge_fixed,
    Ham_Psi_scaled,
)
from NES_VMC_tool import make_MS_estimator_fn,compute_lam_v_from_samples,level_to_active_col
from H2_631G import SINGLE_SIZE, ha, hi, K, Hatree_Fock, E_fcis, hi_ext

HISTORY_FILE = './data/26-09-10-16-25_history_natural_gradient_H2_molecule_K4.pkl'
with open(HISTORY_FILE, 'rb') as f:
    history = pickle.load(f)

print('history keys:', list(history.keys()))
print(f'K = {K}, 单态构型数 = {hi.n_states}, FCI 能级 = {np.round(E_fcis, 5)}')
print(f'samples shape = {history["samples"][-1].shape}')

params = history['params'][-1]
samples = np.asarray(history['samples'][-1])          # (N, K*SINGLE)，整数构型
N = samples.shape[0]
x_batch = samples.reshape(N, K, SINGLE_SIZE)           # (N, K, 8)，保留整数给算符/machine
print(f'重排后 walker 构型 x_batch.shape = {x_batch.shape}（N, K, n_spin）')

∣NK⟩ Tip: You must cite NetKet according to our policy. Use nk.cite() to find out how.

ModuleNotFoundError: No module named 'H2_631G'

## 1. 重建单 walker 的 K×K 矩阵 $P$ 与 $H\psi$

逐列拟设作用到已采样的副本构型上：$P_{n\alpha\beta}=\psi_\beta(x_{n\alpha})$，
并用 $(H\psi)_{n\alpha\beta}=(H\psi_\beta)(x_{n\alpha})$ 由求和路径弹算 $(H\psi)$（不构造 16×16 矩阵）。

In [ ]:
# 用与训练同构的总拟设拿 graphdef，并构造列级 gauge-fixed machine（与训练一致）
total_ansatz = NESTotalAnsatz_stable(
    n_spin_orbitals=SINGLE_SIZE, n_states=K, hidden_dim=SINGLE_SIZE + K,
    rngs=nnx.Rngs(11),
)
single_machine_list = [
    create_single_machine_gauge_fixed(ansatz, Hatree_Fock)[0]
    for ansatz in total_ansatz.single_ansatz_list
]

x_jax = jnp.asarray(x_batch)
# P_{nαβ} = ψ_β(x_{nα})：第 β 列 macheine 对 (N,K,8) 求值 -> (N,K)
logP_cols = [
    single_machine_list[j](params['single_ansatz_list'][j], x_jax)
    for j in range(K)
]
logP = jnp.stack(logP_cols, axis=-1)                 # (N, α, β)

# 每 walker 的全局安全 shift（只做数值稳定；作为每 walker 的公共标度会在归一化里抵消）
shift_n = jnp.max(logP.real, axis=(1, 2)).reshape(-1)
P = jnp.exp(logP - shift_n.reshape(-1, 1, 1))         # (N, K, K)

HP = Ham_Psi_scaled(
    ha=ha,
    single_machine_list=single_machine_list,
    total_params=params,
    x=x_jax,
    shift=shift_n,
)

finite = (
    jnp.all(jnp.isfinite(P), axis=(-2, -1))
    & jnp.all(jnp.isfinite(HP), axis=(-2, -1))
)
print(f'P.shape = {P.shape}（N, K, K） | HP.shape = {HP.shape}（N, K, K）')
print(f'有效 walker 比例 = {float(finite.mean()):.3f}')

In [ ]:
estimator = make_MS_estimator_fn(ha, single_machine_list)
Mhat, Shat, finite = estimator(params, x_batch)

In [ ]:
lam_real, v, n_valid , order = compute_lam_v_from_samples(ha=ha,
                           single_machine_list=single_machine_list,
                           total_params=params,
                           x_batch=x_batch,
                           estimator=estimator)
lam_real